# L4 38 — train the matched-layout Qwen 3B link

Runs the single reserved repair iteration. The sender still uses contextual benign prefixes, but teacher message-token embeddings and student mapped states now occupy the exact same position after an identical receiver prefix.

The adapter initializes from the original faithful link. No game prompts, histories, actions, rewards, or arena results enter training. Base-model weights remain frozen, and Drive checkpoints make the run resumable.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = '77d22d8eb67a00f0f3aad698c21f13b5a6c827b8'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
SOURCE_JOB_ID = 'faithful-qwen3b-t4-001'
JOB_ID = 'matched-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
STEPS = 1000
ROOT = pathlib.Path('/content/drive/MyDrive/rival-arena-l4')
SOURCE_LINK = ROOT/SOURCE_JOB_ID/'faithful_link.pt'
JOB_DIR = ROOT/JOB_ID
JOB_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE_LINK.exists(), 'Missing original trained link in Drive'
print('Output:', JOB_DIR)

In [ ]:
command = [
    'python', 'scripts/train_context_link.py',
    '--model', MODEL,
    '--output', JOB_DIR,
    '--job-id', JOB_ID,
    '--init-link', SOURCE_LINK,
    '--steps', str(STEPS),
    '--gradient-checkpointing',
    '--matched-receiver-layout',
]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected output: `MyDrive/rival-arena-l4/matched-qwen3b-t4-001/`. This is the sole reserved repair lineage. It must pass a fresh matched-layout gate before any pilot.